In [19]:
import os
import time
import numpy as np
import tensorflow as tf

from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input
from tensorflow.keras.models import Model

DATASET_PATH = r"D:\Project Jupyter\MODELLING\Dataset_baruPR"

train_dir = os.path.join(DATASET_PATH, "train")
val_dir   = os.path.join(DATASET_PATH, "val")
test_dir  = os.path.join(DATASET_PATH, "test")

CLASSES = ["edible", "poisonous"]

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS_FINE_TUNE = 5

PRUNE_STAGES = [0.20, 0.30, 0.40]  # ITERATIVE

def preprocess_with_pad(image):
    image = tf.image.resize_with_pad(image, IMG_SIZE, IMG_SIZE)
    image = preprocess_input(image)
    return image

train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_with_pad,
    rotation_range=10,
    zoom_range=0.1,
    brightness_range=[0.7, 1.3],
    horizontal_flip=True
)

val_test_gen = ImageDataGenerator(
    preprocessing_function=preprocess_with_pad
)

attention_gen = ImageDataGenerator(
    preprocessing_function=preprocess_with_pad
)

train_data = train_gen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

val_data = val_test_gen.flow_from_directory(
    val_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_data = val_test_gen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

attention_data = attention_gen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

BASE_MODEL_PATH = r"D:\Project Jupyter\FINAL QUANTIZATION FIX\HASIL MODELBASE\mobilenetv3Large_jamur25COPYBARUFIXTF2PREPRO_tf"


Found 2256 images belonging to 2 classes.
Found 282 images belonging to 2 classes.
Found 232 images belonging to 2 classes.
Found 2256 images belonging to 2 classes.


In [21]:
base_model = tf.keras.models.load_model(
    BASE_MODEL_PATH,
    compile=False
)

base_model.trainable = True
base_model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 MobilenetV3large (Functiona  (None, 7, 7, 960)        2996352   
 l)                                                              
                                                                 
 global_average_pooling2d (G  (None, 960)              0         
 lobalAveragePooling2D)                                          
                                                                 
 dense (Dense)               (None, 128)               123008    
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 dense_1 (Dense)             (None, 2)                 258       
                                                                 
Total params: 3,119,618
Trainable params: 3,095,218
Non-

In [ ]:
se_layers = [
    l for l in model.layers
    if "se" in l.name.lower() and len(l.output_shape) == 2
]

attention_model = Model(
    model.input,
    [l.output for l in se_layers]
)


In [ ]:
def compute_se_importance(attention_model, data_gen, steps=20):
    importance = [None] * len(attention_model.outputs)

    for _ in range(steps):
        x, _ = next(data_gen)
        outputs = attention_model(x, training=False)

        for i, out in enumerate(outputs):
            score = tf.reduce_mean(out, axis=0).numpy()
            importance[i] = score if importance[i] is None else importance[i] + score

    return [imp / steps for imp in importance]


In [ ]:
def attention_guided_prune(model, attention_model, data_gen, prune_ratio):

    se_scores = compute_se_importance(attention_model, data_gen)
    conv_layers = [l for l in model.layers if isinstance(l, Conv2D)]

    for conv, score in zip(conv_layers, se_scores):
        threshold = np.percentile(score, prune_ratio * 100)
        mask = score > threshold

        weights = conv.get_weights()
        kernel = weights[0]

        # 🔥 SOFT PRUNING (AMAN)
        kernel[:, :, :, ~mask] = 0

        if len(weights) > 1:
            conv.set_weights([kernel, weights[1]])
        else:
            conv.set_weights([kernel])

    return model


In [ ]:
from tensorflow.keras.layers import Conv2D, DepthwiseConv2D

model = tf.keras.models.load_model(BASE_MODEL_PATH, compile=False)
model.trainable = True

for ratio in PRUNE_STAGES:
    print(f"\n===== ITERATIVE PRUNE {int(ratio*100)}% =====")

    # 🔁 BANGUN ULANG ATTENTION MODEL (WAJIB)
    se_layers = [
        l for l in model.layers
        if "se" in l.name.lower() and len(l.output_shape) == 2
    ]

    attention_model = Model(
        inputs=model.input,
        outputs=[l.output for l in se_layers]
    )

    # ✂️ PRUNE
    model = attention_guided_prune(
        model,
        attention_model,
        attention_data,
        ratio
    )

   


In [ ]:
import time

model.compile(
        optimizer=Adam(1e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )



start_time = time.time()

model.fit(
        train_data,
        validation_data=val_data,
        epochs=EPOCHS_FINE_TUNE,
        verbose=1
    )

end_time = time.time()

training_time = end_time - start_time

print(f"\nTotal Training Time : {training_time:.2f} seconds")
print(f"Total Training Time : {training_time/60:.2f} minutes")
print(f"Avg Time / Epoch    : {training_time/EPOCHS_FINE_TUNE:.2f} seconds")


In [ ]:
PRUNED_MODEL_PATH = r"D:\Project Jupyter\mobilenetv3_se_attention_pruned_final20_30_40FINALFIX"
model.save(
    PRUNED_MODEL_PATH,
    include_optimizer=False
)


In [ ]:
import os

def get_model_size_mb(model_path):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(model_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total_size += os.path.getsize(fp)
    return total_size / (1024 * 1024)


In [ ]:
base_model_size = get_model_size_mb(BASE_MODEL_PATH)
pruned_model_size = get_model_size_mb(PRUNED_MODEL_PATH)

print(f"Ukuran Baseline Model : {base_model_size:.2f} MB")
print(f"Ukuran Pruned Model   : {pruned_model_size:.2f} MB")

compression_ratio = base_model_size / pruned_model_size
size_reduction = (1 - (pruned_model_size / base_model_size)) * 100

print(f"Compression Ratio : {compression_ratio:.2f}x")
print(f"Size Reduction    : {size_reduction:.2f}%")


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix


In [ ]:
test_data.reset()


In [ ]:
predictions = model.predict(test_data, verbose=1)
y_pred = np.argmax(predictions, axis=1)
y_true = test_data.classes


In [ ]:
print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=CLASSES
))


In [ ]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))


In [ ]:
import time
import numpy as np

def benchmark_inference(model, data_gen, warmup=3, runs=10):
    """
    model      : final pruned model (tanpa pruning wrapper)
    data_gen   : test_data (shuffle=False)
    """

    # Ambil 1 batch
    x_batch, _ = next(data_gen)

    # Warm-up (penting)
    for _ in range(warmup):
        _ = model.predict(x_batch, verbose=0)

    times = []

    for _ in range(runs):
        start = time.time()
        _ = model.predict(x_batch, verbose=0)
        end = time.time()
        times.append(end - start)

    avg_time = np.mean(times)
    per_image_time = avg_time / x_batch.shape[0]

    print("\nInference Benchmark (CPU)")
    print(f"Batch size         : {x_batch.shape[0]}")
    print(f"Avg batch time     : {avg_time:.4f} seconds")
    print(f"Avg per image time : {per_image_time:.6f} seconds")

    return avg_time, per_image_time


In [ ]:
benchmark_inference(base_model, test_data)
